# 01. Data Audit - Amazon Sale Report (Q2 2022)

Audit menyeluruh terhadap dataset Amazon_Sale_Report_Cleaned.csv untuk keperluan Machine Learning.

In [1]:
import pandas as pd
import numpy as np

csv_path = '../data/raw/Amazon_Sale_Report_Cleaned.csv'
df = pd.read_csv(csv_path, low_memory=False)
print('Shape:', df.shape)
df.head()


Shape: (42357, 29)


,index,Order ID,Status,Fulfilment,Sales Channel,ship-service-level,Style,SKU,Category,Size,...,promotion-ids,B2B,fulfilled-by,Unnamed: 22,order_date,year,month,month_name,is_cancelled,is_delivered
0,0,405-8078784-5731545,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,...,NaN,False,Easy Ship,NaN,2022-04-30,2022,2022-04,Apr 2022,True,False
1,1,171-9198151-1101146,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,Kurta,3XL,...,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship,NaN,2022-04-30,2022,2022-04,Apr 2022,False,True
2,2,404-0687676-7273146,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,Kurta,XL,...,IN Core Free Shipping 2015/04/08 23-48-5-108,True,NaN,NaN,2022-04-30,2022,2022-04,Apr 2022,False,False
3,3,403-9615377-8133951,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,...,NaN,False,Easy Ship,NaN,2022-04-30,2022,2022-04,Apr 2022,True,False
4,4,407-1069790-7240320,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,...,NaN,False,NaN,NaN,2022-04-30,2022,2022-04,Apr 2022,False,False


## 1. Pemeriksaan Struktur, Tipe Data, dan Nilai Kosong

In [2]:
null_info = pd.DataFrame({
    'dtype': df.dtypes,
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
null_info[null_info['null_count'] > 0]


,dtype,null_count,null_pct
Fulfilment,str,1,0.00
Sales Channel,str,1,0.00
ship-service-level,str,1,0.00
Style,str,1,0.00
SKU,str,1,0.00
ASIN,str,1,0.00
Courier Status,str,2514,5.94
currency,str,2566,6.06
ship-city,str,14,0.03
ship-state,str,14,0.03


## 2. Pemeriksaan Duplikasi dan Order ID

In [3]:
full_dups = df.duplicated().sum()
unique_orders = df['Order ID'].nunique()
dup_orders = df['Order ID'].duplicated().sum()
print(f'Full Duplicates: {full_dups}')
print(f'Unique Order IDs: {unique_orders}')
print(f'Duplicate Order IDs (multi-item orders): {dup_orders}')


Full Duplicates: 0
Unique Order IDs: 39604
Duplicate Order IDs (multi-item orders): 2753


## 3. Rentang Tanggal Transaksi

In [4]:
df['parsed_date'] = pd.to_datetime(df['order_date'], errors='coerce')
print('Min Date:', df['parsed_date'].min())
print('Max Date:', df['parsed_date'].max())
print('Total Active Days:', df['parsed_date'].dt.date.nunique())
print('Distribusi per Bulan:\n', df['month'].value_counts())


Min Date: 2022-04-05 00:00:00
Max Date: 2022-04-30 00:00:00
Total Active Days: 26
Distribusi per Bulan:
 month
2022-04    42356
Name: count, dtype: int64


## 4. Nilai Unik dan Frekuensi Kolom Kategorikal Utama

In [5]:
cat_cols = ['Status', 'Fulfilment', 'Category', 'Size', 'ship-service-level', 'B2B']
for col in cat_cols:
    print(f'=== {col} ===')
    print(df[col].value_counts(dropna=False))
    print()


=== Status ===
Status
Shipped                          24519
Shipped - Delivered to Buyer     10871
Cancelled                         6132
Shipped - Returned to Seller       820
Shipped - Rejected by Buyer          5
Shipped - Picked Up                  4
Shipped - Lost in Transit            2
Shipped - Returning to Seller        2
Shipped - Out for Delivery           1
Unknown                              1
Name: count, dtype: int64

=== Fulfilment ===
Fulfilment
Amazon      28142
Merchant    14214
NaN             1
Name: count, dtype: int64

=== Category ===
Category
Set              17253
Kurta            17161
Western Dress     3738
Top               3291
Blouse             370
Ethnic Dress       309
Bottom             167
Saree               67
Other                1
Name: count, dtype: int64

=== Size ===
Size
L          7449
M          7439
XL         6821
XXL        5839
S          5803
3XL        4914
XS         3810
FREE        172
6XL          44
5XL          38
4XL         

## 5. Distribusi Amount (Outlier & Nilai 0)

In [6]:
amt = df['Amount']
print('Amount Summary:')
print(amt.describe())
print(f'Zero Amount Count: {(amt == 0).sum()}')
print(f'Negative Amount Count: {(amt < 0).sum()}')

q1 = amt.quantile(0.25)
q3 = amt.quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
lower_bound = q1 - 1.5 * iqr
outliers = df[(amt < lower_bound) | (amt > upper_bound)]
print(f'IQR: {iqr}, Upper Bound: {upper_bound}, Lower Bound: {lower_bound}')
print(f'Outliers Count (Amount > {upper_bound}): {len(outliers)}')


Amount Summary:
count    42357.000000
mean       586.802275
std        303.106571
min          0.000000
25%        399.000000
50%        549.000000
75%        759.000000
max       5584.000000
Name: Amount, dtype: float64
Zero Amount Count: 3284
Negative Amount Count: 0
IQR: 360.0, Upper Bound: 1299.0, Lower Bound: -141.0
Outliers Count (Amount > 1299.0): 634


## 6. Distribusi Qty

In [7]:
qty = df['Qty']
print(qty.value_counts(dropna=False))
print(f'Zero Qty Count: {(qty == 0).sum()}')


Qty
1     37804
0      4423
2       108
3        13
4         4
15        1
9         1
13        1
5         1
8         1
Name: count, dtype: int64
Zero Qty Count: 4423


## 7. Variasi Ejaan ship-state

In [8]:
raw_states = df['ship-state'].dropna().unique()
print(f'Unique raw states count: {len(raw_states)}')
print(sorted([str(s) for s in raw_states]))


Unique raw states count: 56
['ANDAMAN & NICOBAR ', 'ANDHRA PRADESH', 'ARUNACHAL PRADESH', 'ASSAM', 'BIHAR', 'Bihar', 'CHANDIGARH', 'CHHATTISGARH', 'Chandigarh', 'DADRA AND NAGAR', 'DELHI', 'Delhi', 'GOA', 'Goa', 'Gujarat', 'HARYANA', 'HIMACHAL PRADESH', 'JAMMU & KASHMIR', 'JHARKHAND', 'KARNATAKA', 'KERALA', 'LADAKH', 'LAKSHADWEEP', 'MADHYA PRADESH', 'MAHARASHTRA', 'MANIPUR', 'MEGHALAYA', 'MIZORAM', 'Manipur', 'NAGALAND', 'NL', 'New Delhi', 'ODISHA', 'Odisha', 'PB', 'PUDUCHERRY', 'PUNJAB', 'Puducherry', 'Punjab', 'Punjab/Mohali/Zirakpur', 'RAJASTHAN', 'RJ', 'Rajasthan', 'Rajshthan', 'SIKKIM', 'TAMIL NADU', 'TELANGANA', 'TRIPURA', 'UTTAR PRADESH', 'UTTARAKHAND', 'WEST BENGAL', 'delhi', 'goa', 'orissa', 'punjab', 'rajasthan']


## 8. Definisi Target is_cancelled

In [9]:
print('Frekuensi Status Lengkap:')
print(df['Status'].value_counts())
print('\nKorelasi Status dengan is_cancelled di dataset:')
print(pd.crosstab(df['Status'], df['is_cancelled']))


Frekuensi Status Lengkap:
Status
Shipped                          24519
Shipped - Delivered to Buyer     10871
Cancelled                         6132
Shipped - Returned to Seller       820
Shipped - Rejected by Buyer          5
Shipped - Picked Up                  4
Shipped - Lost in Transit            2
Shipped - Returning to Seller        2
Shipped - Out for Delivery           1
Unknown                              1
Name: count, dtype: int64

Korelasi Status dengan is_cancelled di dataset:
is_cancelled                   False  True 
Status                                     
Cancelled                          0   6132
Shipped                        24519      0
Shipped - Delivered to Buyer   10871      0
Shipped - Lost in Transit          2      0
Shipped - Out for Delivery         1      0
Shipped - Picked Up                4      0
Shipped - Rejected by Buyer        5      0
Shipped - Returned to Seller     820      0
Shipped - Returning to Seller      2      0
Unknown           